# 24 — Correlation Analysis

Explore how features, sensors, and signals relate to each other across the bearing lifetime.

**Dataset**: first study in any ISA-PHM dataset  
**API**: `assay.plot_correlation()` · `study.plot_sensor_lifecycle_correlation()` · `study.plot_cross_correlation()`
Run the import and dataset-loading cells first. Each following example selects what it needs independently; change its study, assay, or run to explore your data.


## Imports and dataset path


In [ ]:
from pathlib import Path

from isa_phm import ISAWrapper
from bokeh.io import output_notebook
from bokeh.plotting import show as bokeh_show

output_notebook()

# Set this to your dataset's ISA-PHM JSON file.
ISA_JSON = Path("path/to/your-dataset.isa-phm.json").expanduser()
ISA_JSON = Path("path/to/your-dataset.isa-phm.json").expanduser()
DATA_ROOT = ISA_JSON.parent

print("ISA-JSON :", ISA_JSON)
print("DATA_ROOT:", DATA_ROOT)
print("Exists   :", ISA_JSON.exists())


## Load the wrapper


In [ ]:
wrapper = ISAWrapper(
    ISA_JSON,
    data_root=DATA_ROOT,
    strict_validation=False,
    cache_maxsize=10,
)


## Dataset overview


In [ ]:
study = wrapper.study(1)  # first study (1-based index)
assay = study.assay(1)   # horizontal accelerometer

print(f"Study  : {study.title}")
print(f"Assay  : {assay.assay_id}  ({assay.run_count} runs)")
print(f"Sensors: {', '.join(a.assay_id for a in study.list_assays())}")


## 1. Feature correlation across runs

`plot_correlation()` computes the Pearson correlation between all lifecycle feature pairs across all runs.  
RMS, std, peak2peak, and max typically cluster together.  
Kurtosis and crest_factor often anti-correlate with RMS in early stages — the classic *kurtosis reversal*.

In [ ]:
study = wrapper.study(1)  # first study (1-based index)
assay = study.assay(1)

fig = assay.plot_correlation(file_type="raw", n_workers=8)
bokeh_show(fig)

## 2. Sensor-to-sensor lifecycle correlation

`study.plot_sensor_lifecycle_correlation()` builds a matrix where each row is a run and each column is a sensor.  
The heatmap shows how the RMS of the horizontal channel correlates with the vertical channel — and whether both degrade simultaneously.

In [ ]:
study = wrapper.study(1)  # first study (1-based index)

# Compare RMS degradation across both sensor channels
fig = study.plot_sensor_lifecycle_correlation(
    feature="rms",
    file_type="raw",
    n_workers=8,
)
bokeh_show(fig)

In [ ]:
study = wrapper.study(1)  # first study (1-based index)

# Kurtosis — often shows an earlier correlation break between channels
fig = study.plot_sensor_lifecycle_correlation(
    feature="kurtosis",
    file_type="raw",
    n_workers=8,
)
bokeh_show(fig)

## 3. Cross-correlation between two sensors

`plot_cross_correlation()` shows the normalised cross-correlation between two sensors as a function of sample lag (±`max_lag`).  
A peak at lag 0 means the two sensors respond simultaneously.  
A non-zero peak lag indicates one sensor leads the other by that many samples.

In [ ]:
study = wrapper.study(1)  # first study (1-based index)
assay = study.assay(1)

# Healthy run — horizontal vs vertical channel
first_run = assay.run(1)

fig = study.plot_cross_correlation(
    assay_id_1=1,
    assay_id_2=2,
    run_id=first_run.run_id,
    max_lag=200,
    file_type="raw",
    title=f"Cross-correlation — Healthy (run {first_run.run_number})",
)
bokeh_show(fig)

In [ ]:
study = wrapper.study(1)  # first study (1-based index)
assay = study.assay(1)

# Near-failure run — does the lag change?
last_run = assay.list_runs()[-1]

fig = study.plot_cross_correlation(
    assay_id_1=1,
    assay_id_2=2,
    run_id=last_run.run_id,
    max_lag=200,
    file_type="raw",
    title=f"Cross-correlation — Near Failure (run {last_run.run_number})",
)
bokeh_show(fig)